# 07 · Chunking under a fixed evidence budget

## The engineering question
Does a chunking change retrieve better evidence, or simply send more text to the reader?

**Learning outcomes:** compare fixed and structural boundaries; expand child matches to parent documents; hold gold source spans constant; compare fixed-k and fixed-token budgets; inspect overlap and truncation costs.

A chunk is a view over canonical source text. Its boundaries should not define the gold label. We compare several deterministic strategies on the approved development questions using the same encoder and the same 180-token context budget. Parent/child retrieval ranks small child chunks but expands to a parent source before packing; it can improve context while consuming more tokens.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("07", "B", docker=False)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from vector_masterclass.data import documents, queries, validate_gold
from vector_masterclass.chunking import chunk_document, pack_chunks
from vector_masterclass.models import embed, dense_model
from vector_masterclass.retrieval import exact_search
from vector_masterclass.metrics import ranking_metrics, evidence_coverage

assert validate_gold()["human_review"] == "approved"
docs = documents(tenant="public")
by_key = {d.key: d for d in docs}
questions = [q for q in queries("development") if q.answerable]
query_vectors = embed([q.text for q in questions], role="query")
tokenizer = dense_model().tokenizer
configurations = [
    ("fixed-small", "fixed", 16, 4),
    ("fixed-large", "fixed", 48, 8),
    ("structural", "structural", 48, 8),
    ("parent-child", "parent_child", 16, 4),
]
rows = []
for label, strategy, words, overlap in configurations:
    chunks = [c for doc in docs for c in chunk_document(doc, strategy, words, overlap)]
    lookup = {c.id: c for c in chunks}
    vectors = embed([c.text for c in chunks])
    for q, vector in zip(questions, query_vectors):
        ranked = exact_search(vector, vectors, list(lookup), k=3)
        retrieved = [lookup[id_] for id_, score in ranked]
        if strategy == "parent_child":
            retrieved = [chunk_document(by_key[c.document_key], "document")[0] for c in retrieved]
        fixed_k = evidence_coverage(retrieved, q.evidence)["coverage"]
        packed = pack_chunks(retrieved, tokenizer, budget=180)
        packed_coverage = evidence_coverage(packed["chunks"], q.evidence)["coverage"]
        metrics = ranking_metrics([c.document_key for c in retrieved], q.relevance, k=3)
        rows.append(
            {
                "strategy": label,
                "query": q.id,
                "chunk_count": len(chunks),
                "fixed_k_coverage": fixed_k,
                "fixed_token_coverage": packed_coverage,
                "context_tokens": packed["tokens"],
                "document_recall": metrics["recall"],
                "duplicate_document_fraction": 1
                - len({c.document_key for c in retrieved}) / len(retrieved),
            }
        )
results = lesson.table(pd.DataFrame(rows))
display(results)
summary = results.groupby("strategy")[
    ["fixed_k_coverage", "fixed_token_coverage", "context_tokens"]
].mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
summary[["fixed_k_coverage", "fixed_token_coverage"]].rename(
    columns={"fixed_k_coverage": "Fixed k", "fixed_token_coverage": "Fixed token budget"}
).plot.bar(ax=axes[0])
axes[0].set(
    ylabel="Fraction of evidence units covered",
    ylim=(0, 1.1),
    title="Context quantity changes the comparison",
)
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, 1.3), ncol=2)
summary.context_tokens.plot.bar(ax=axes[1], color="#2563eb")
axes[1].axhline(180, linestyle="--", color="black")
axes[1].set(ylabel="Mean packed tokens", title="Whole-chunk packing includes citation headers")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Parent expansion can lose under a small context budget when a whole parent does not fit. Small chunks can miss a qualifier; large chunks can dilute the embedding or be truncated. The correct decision depends on source structure, query type, and the downstream context budget. This small development corpus is a controlled demonstration, not a universal chunk-size recommendation.

**Exercise:** halve the token budget without changing k. Inspect one query where the ranking stays the same but evidence coverage changes. Then vary overlap and verify that repeated source characters never earn duplicate gold credit.

The default chunking configuration remains fixed for subsequent lessons; this notebook does not tune against frozen test questions. Generated contextual prefixes are introduced separately in the live-RAG track because they add model cost and can introduce unsupported claims.

In [ ]:
manifest = lesson.finish(
    {
        "all_strategies_measured": len(set(results.strategy)) == 4,
        "context_budget_enforced": bool((results.context_tokens <= 180).all()),
        "coverage_bounded": bool(
            results[["fixed_k_coverage", "fixed_token_coverage"]].le(1).all().all()
        ),
        "gold_unchanged": validate_gold()["human_review"] == "approved",
    },
    observations={
        "configuration_count": len(configurations),
        "queries": len(questions),
        "token_budget": 180,
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.